GGNN Model


In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import json

PROJECT_DIR = "/content/drive/MyDrive/TAOD_GGNN"
os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_PATH = f"{PROJECT_DIR}/samples.jsonl"

samples = [
    {
        "image_id": "img001",
        "task_id": 1,
        "image_width": 640,
        "image_height": 640,
        "detections": [
            {
                "bbox": [100, 120, 300, 500],
                "class_id": 56,
                "class_name": "chair",
                "conf": 0.91
            },
            {
                "bbox": [350, 180, 500, 430],
                "class_id": 60,
                "class_name": "dining table",
                "conf": 0.77
            },
            {
                "bbox": [40, 300, 120, 420],
                "class_id": 39,
                "class_name": "bottle",
                "conf": 0.82
            }
        ],
        "target_index": 0
    },

    {
        "image_id": "img002",
        "task_id": 2,
        "image_width": 640,
        "image_height": 640,
        "detections": [
            {
                "bbox": [90, 100, 240, 300],
                "class_id": 39,
                "class_name": "bottle",
                "conf": 0.93
            },
            {
                "bbox": [250, 200, 390, 530],
                "class_id": 41,
                "class_name": "cup",
                "conf": 0.88
            },
            {
                "bbox": [420, 180, 560, 440],
                "class_id": 56,
                "class_name": "chair",
                "conf": 0.75
            }
        ],
        "target_index": 1
    }
]

with open(DATA_PATH, "w") as f:
    for sample in samples:
        f.write(json.dumps(sample) + "\n")

print("Created:", DATA_PATH)

Created: /content/drive/MyDrive/TAOD_GGNN/samples.jsonl


In [17]:
with open(DATA_PATH, "r") as f:
    for line in f:
        print(line)

{"image_id": "img001", "task_id": 1, "image_width": 640, "image_height": 640, "detections": [{"bbox": [100, 120, 300, 500], "class_id": 56, "class_name": "chair", "conf": 0.91}, {"bbox": [350, 180, 500, 430], "class_id": 60, "class_name": "dining table", "conf": 0.77}, {"bbox": [40, 300, 120, 420], "class_id": 39, "class_name": "bottle", "conf": 0.82}], "target_index": 0}

{"image_id": "img002", "task_id": 2, "image_width": 640, "image_height": 640, "detections": [{"bbox": [90, 100, 240, 300], "class_id": 39, "class_name": "bottle", "conf": 0.93}, {"bbox": [250, 200, 390, 530], "class_id": 41, "class_name": "cup", "conf": 0.88}, {"bbox": [420, 180, 560, 440], "class_id": 56, "class_name": "chair", "conf": 0.75}], "target_index": 1}



In [18]:
import os
import json
import random
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Torch: 2.10.0+cpu
CUDA available: False


In [19]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/TAOD_GGNN"
DATA_PATH = f"{PROJECT_DIR}/samples.jsonl"
SAVE_DIR = f"{PROJECT_DIR}/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Data path:", DATA_PATH)
print("Save dir:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data path: /content/drive/MyDrive/TAOD_GGNN/samples.jsonl
Save dir: /content/drive/MyDrive/TAOD_GGNN/checkpoints


In [20]:
NUM_COCO_CLASSES = 80
NUM_TASKS = 14

OBJECT_FEATURE_DIM = 87
TASK_FEATURE_DIM = 14
INPUT_DIM = OBJECT_FEATURE_DIM + TASK_FEATURE_DIM

MAX_NODES = 60


def load_jsonl(path):
    samples = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
    return samples


def normalize_task_id(task_id):
    """
    Accepts task_id as 1..14 or 0..13.
    Internally returns 0..13.
    """
    task_id = int(task_id)
    if 1 <= task_id <= NUM_TASKS:
        return task_id - 1
    if 0 <= task_id < NUM_TASKS:
        return task_id
    raise ValueError(f"Invalid task_id: {task_id}")


def bbox_to_geometry_features(bbox, image_width=640, image_height=640):
    """
    Converts bbox [x1,y1,x2,y2] into 6 geometry features:
    cx, cy, w, h, area, log_aspect
    All normalized/stabilized.
    """
    x1, y1, x2, y2 = map(float, bbox)

    # If bbox already seems normalized, keep it.
    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.5:
        nx1, ny1, nx2, ny2 = x1, y1, x2, y2
    else:
        nx1 = x1 / image_width
        ny1 = y1 / image_height
        nx2 = x2 / image_width
        ny2 = y2 / image_height

    nx1, ny1, nx2, ny2 = np.clip([nx1, ny1, nx2, ny2], 0.0, 1.0)

    w = max(nx2 - nx1, 1e-6)
    h = max(ny2 - ny1, 1e-6)
    cx = nx1 + w / 2.0
    cy = ny1 + h / 2.0
    area = w * h

    aspect = w / h
    log_aspect = np.log(aspect + 1e-6)
    log_aspect = np.clip(log_aspect, -3.0, 3.0) / 3.0

    return np.array([cx, cy, w, h, area, log_aspect], dtype=np.float32)


def detection_to_node_feature(det, task_id, image_width=640, image_height=640):
    geom = bbox_to_geometry_features(
        det["bbox"],
        image_width=image_width,
        image_height=image_height
    )

    conf = np.array([float(det.get("conf", 1.0))], dtype=np.float32)

    class_id = int(det["class_id"])
    class_onehot = np.zeros(NUM_COCO_CLASSES, dtype=np.float32)
    if 0 <= class_id < NUM_COCO_CLASSES:
        class_onehot[class_id] = 1.0
    else:
        raise ValueError(f"Invalid COCO class_id: {class_id}")

    object_feature = np.concatenate([geom, conf, class_onehot], axis=0)

    task_id = normalize_task_id(task_id)
    task_onehot = np.zeros(NUM_TASKS, dtype=np.float32)
    task_onehot[task_id] = 1.0

    full_feature = np.concatenate([object_feature, task_onehot], axis=0)

    assert full_feature.shape[0] == INPUT_DIM
    return full_feature


class TAODGraphDataset(Dataset):
    def __init__(self, jsonl_path, max_nodes=60):
        self.samples = load_jsonl(jsonl_path)
        self.max_nodes = max_nodes

        valid_samples = []
        skipped = 0

        for s in self.samples:
            if "detections" not in s or len(s["detections"]) == 0:
                skipped += 1
                continue
            if "target_index" not in s:
                skipped += 1
                continue
            if int(s["target_index"]) < 0 or int(s["target_index"]) >= len(s["detections"]):
                skipped += 1
                continue
            valid_samples.append(s)

        self.samples = valid_samples
        print(f"Loaded {len(self.samples)} valid samples. Skipped {skipped} invalid samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        detections = s["detections"]
        target_index_original = int(s["target_index"])

        image_width = int(s.get("image_width", 640))
        image_height = int(s.get("image_height", 640))
        task_id = s["task_id"]

        # Sort detections by confidence but keep target object.
        indexed_dets = list(enumerate(detections))
        indexed_dets = sorted(
            indexed_dets,
            key=lambda x: float(x[1].get("conf", 1.0)),
            reverse=True
        )

        chosen = indexed_dets[:self.max_nodes]

        chosen_original_indices = [x[0] for x in chosen]

        if target_index_original not in chosen_original_indices:
            chosen[-1] = (target_index_original, detections[target_index_original])
            chosen_original_indices = [x[0] for x in chosen]

        target_index_new = chosen_original_indices.index(target_index_original)

        node_features = []
        kept_detections = []

        for original_idx, det in chosen:
            feat = detection_to_node_feature(
                det,
                task_id=task_id,
                image_width=image_width,
                image_height=image_height
            )
            node_features.append(feat)
            kept_detections.append(det)

        x = torch.tensor(np.stack(node_features), dtype=torch.float32)
        y = torch.tensor(target_index_new, dtype=torch.long)

        return {
            "x": x,
            "y": y,
            "num_nodes": x.shape[0],
            "image_id": s.get("image_id", str(idx)),
            "task_id": task_id,
            "detections": kept_detections
        }


def collate_graphs(batch):
    batch_size = len(batch)
    max_n = max(item["num_nodes"] for item in batch)

    x = torch.zeros(batch_size, max_n, INPUT_DIM, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_n, dtype=torch.bool)
    adj = torch.zeros(batch_size, max_n, max_n, dtype=torch.float32)
    y = torch.zeros(batch_size, dtype=torch.long)

    metadata = []

    for b, item in enumerate(batch):
        n = item["num_nodes"]

        x[b, :n] = item["x"]
        mask[b, :n] = True
        y[b] = item["y"]

        # Fully connected object graph, excluding self-loops.
        if n > 1:
            adj[b, :n, :n] = 1.0
            adj[b, torch.arange(n), torch.arange(n)] = 0.0
            adj[b, :n, :n] = adj[b, :n, :n] / (n - 1)

        metadata.append({
            "image_id": item["image_id"],
            "task_id": item["task_id"],
            "detections": item["detections"]
        })

    return {
        "x": x,
        "adj": adj,
        "mask": mask,
        "y": y,
        "metadata": metadata
    }

In [21]:
class GGNNRanker(nn.Module):
    def __init__(self, input_dim=101, hidden_dim=128, num_steps=4, dropout=0.15):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_steps = num_steps

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.message_layer = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        self.score_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, adj, mask):
        """
        x:    [B, N, F]
        adj:  [B, N, N]
        mask: [B, N]
        """
        B, N, F = x.shape

        h = self.input_proj(x)

        for _ in range(self.num_steps):
            messages = self.message_layer(h)
            aggregated = torch.bmm(adj, messages)

            h_flat = h.reshape(B * N, self.hidden_dim)
            agg_flat = aggregated.reshape(B * N, self.hidden_dim)

            updated_flat = self.gru(agg_flat, h_flat)
            updated = updated_flat.reshape(B, N, self.hidden_dim)

            h = torch.where(mask.unsqueeze(-1), updated, h)

        scores = self.score_head(h).squeeze(-1)

        # Invalid padded nodes should never be selected.
        scores = scores.masked_fill(~mask, -1e9)

        return scores

In [22]:
dataset = TAODGraphDataset(DATA_PATH, max_nodes=MAX_NODES)

total = len(dataset)
train_size = int(0.70 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_graphs
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_graphs
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_graphs
)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

Loaded 2 valid samples. Skipped 0 invalid samples.
Train: 1
Val: 0
Test: 1


In [25]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        x = batch["x"].to(DEVICE)
        adj = batch["adj"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        y = batch["y"].to(DEVICE)

        optimizer.zero_grad()

        scores = model(x, adj, mask)
        loss = criterion(scores, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        pred = scores.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_targets = []

    # If loader is empty, return safe values instead of crashing
    if len(loader) == 0:
        return {
            "loss": None,
            "accuracy": None,
            "preds": [],
            "targets": []
        }

    for batch in loader:
        x = batch["x"].to(DEVICE)
        adj = batch["adj"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        y = batch["y"].to(DEVICE)

        scores = model(x, adj, mask)
        loss = criterion(scores, y)

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size

        pred = scores.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += batch_size

        all_preds.extend(pred.cpu().tolist())
        all_targets.extend(y.cpu().tolist())

    if total == 0:
        return {
            "loss": None,
            "accuracy": None,
            "preds": all_preds,
            "targets": all_targets
        }

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "preds": all_preds,
        "targets": all_targets
    }

In [26]:
model = GGNNRanker(
    input_dim=INPUT_DIM,
    hidden_dim=128,
    num_steps=4,
    dropout=0.15
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4
)

EPOCHS = 50
best_val_acc = -1.0
best_path = f"{SAVE_DIR}/best_ggnn_ranker.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    # Validation may be empty when dataset is tiny
    val_result = evaluate(model, val_loader, criterion)

    if val_result["accuracy"] is None:
        val_loss = None
        val_acc = None

        # Use train accuracy only temporarily for dummy/small dataset testing
        save_metric = train_acc
        print_val_loss = "N/A"
        print_val_acc = "N/A"
    else:
        val_loss = val_result["loss"]
        val_acc = val_result["accuracy"]

        save_metric = val_acc
        print_val_loss = f"{val_loss:.4f}"
        print_val_acc = f"{val_acc:.4f}"

        scheduler.step(val_acc)

    if save_metric > best_val_acc:
        best_val_acc = save_metric
        torch.save({
            "model_state_dict": model.state_dict(),
            "input_dim": INPUT_DIM,
            "hidden_dim": 128,
            "num_steps": 4,
            "num_tasks": NUM_TASKS,
            "num_coco_classes": NUM_COCO_CLASSES,
            "best_metric": best_val_acc
        }, best_path)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {print_val_loss} | "
        f"Val Acc: {print_val_acc} | "
        f"Best: {best_val_acc:.4f}"
    )

print("Best model saved to:", best_path)

Epoch 001 | Train Loss: 1.1012 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 0.0000
Epoch 002 | Train Loss: 1.1012 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 0.0000
Epoch 003 | Train Loss: 1.1029 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 0.0000
Epoch 004 | Train Loss: 1.1071 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 0.0000
Epoch 005 | Train Loss: 1.1033 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 0.0000
Epoch 006 | Train Loss: 1.0870 | Train Acc: 1.0000 | Val Loss: N/A | Val Acc: N/A | Best: 1.0000
Epoch 007 | Train Loss: 1.0969 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 1.0000
Epoch 008 | Train Loss: 1.1144 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 1.0000
Epoch 009 | Train Loss: 1.0993 | Train Acc: 0.0000 | Val Loss: N/A | Val Acc: N/A | Best: 1.0000
Epoch 010 | Train Loss: 1.0839 | Train Acc: 1.0000 | Val Loss: N/A | Val Acc: N/A | Best: 1.0000
Epoch 011 | Train Loss: 1.0811

In [27]:
checkpoint = torch.load(best_path, map_location=DEVICE)

best_model = GGNNRanker(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    num_steps=checkpoint["num_steps"],
    dropout=0.0
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])

test_result = evaluate(best_model, test_loader, criterion)

print("Test loss:", test_result["loss"])
print("Test Top-1 accuracy:", test_result["accuracy"])

Test loss: 1.0987024307250977
Test Top-1 accuracy: 0.0


In [28]:
@torch.no_grad()
def predict_best_object(model, sample):
    model.eval()

    temp_path = "/content/temp_one_sample.jsonl"
    with open(temp_path, "w") as f:
        f.write(json.dumps(sample) + "\n")

    temp_dataset = TAODGraphDataset(temp_path, max_nodes=MAX_NODES)
    temp_loader = DataLoader(
        temp_dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=collate_graphs
    )

    batch = next(iter(temp_loader))

    x = batch["x"].to(DEVICE)
    adj = batch["adj"].to(DEVICE)
    mask = batch["mask"].to(DEVICE)

    scores = model(x, adj, mask)
    probs = torch.softmax(scores, dim=1)

    best_idx = scores.argmax(dim=1).item()
    best_score = probs[0, best_idx].item()

    det = batch["metadata"][0]["detections"][best_idx]

    return {
        "best_index": best_idx,
        "best_score": best_score,
        "best_detection": det
    }


# Example:
sample = dataset.samples[0]
result = predict_best_object(best_model, sample)

print("Image:", sample.get("image_id"))
print("Task:", sample["task_id"])
print("Predicted best object index:", result["best_index"])
print("Suitability score:", result["best_score"])
print("Detection:", result["best_detection"])

Loaded 1 valid samples. Skipped 0 invalid samples.
Image: img001
Task: 1
Predicted best object index: 0
Suitability score: 0.334879070520401
Detection: {'bbox': [100, 120, 300, 500], 'class_id': 56, 'class_name': 'chair', 'conf': 0.91}


In [29]:
import os

best_path = "/content/drive/MyDrive/TAOD_GGNN/checkpoints/best_ggnn_ranker.pt"

if os.path.exists(best_path):
    print("GGNN model saved successfully:", best_path)
else:
    print("Model not found. Training may not have saved correctly.")

GGNN model saved successfully: /content/drive/MyDrive/TAOD_GGNN/checkpoints/best_ggnn_ranker.pt
